# 💸 Dompy: Asisten Budget Pribadi Berbasis Gemini

Dompy adalah chatbot pengelola keuangan pribadi. Kamu cukup ngobrol biasa, misalnya *"budget makan minggu ini 300rb"* atau *"tadi makan bakso 18rb"*, lalu Dompy mencatat, menghitung sisa budget, dan memberi masukan di akhir periode.

**Cara kerja singkat**
- **Gemini** bertugas memahami bahasa sehari-hari (nominal seperti "18rb", "goceng", tanggal seperti "kemarin").
- Semua **perhitungan dan penyimpanan** dikerjakan fungsi Python + SQLite yang dipanggil Gemini lewat *function calling*. Jadi angka sisa budget selalu akurat, bukan tebakan model.
- Data tersimpan di database, jadi Dompy tetap ingat budget dan pengeluaranmu walaupun aplikasi ditutup.

**Struktur notebook.** Aplikasi Streamlit harus berupa file `.py`, jadi sebagian besar sel di bawah memakai `%%writefile` untuk menulis file:

| File | Isi |
|---|---|
| `config.py` | Parameter yang bisa diubah (model, temperature, gaya bahasa, dll.) |
| `database.py` | Tabel SQLite dan fungsi bantu (periode, format rupiah) |
| `tools.py` | 7 fungsi yang dipanggil Gemini lewat function calling |
| `agent.py` | Koneksi Gemini, system prompt, dan sesi chat |
| `charts.py` | Grafik Plotly untuk dashboard |
| `app.py` | Tampilan Streamlit |

Jalankan sel dari atas ke bawah. Sel terakhir membuka aplikasinya.

## 1. Instalasi library

In [ ]:
!pip install -q -U google-genai streamlit pandas plotly tzdata

## 2. Konfigurasi

Ubah nilai di bawah sesuai selera, lalu jalankan selnya. Gaya bahasa dan mode saran juga bisa diganti langsung dari sidebar aplikasi.

In [ ]:
%%writefile config.py
import os

MODEL_NAME = "gemini-flash-latest"  # alias Flash terbaru; bisa diganti mis. "gemini-3.6-flash"
MODEL_CADANGAN = ["gemini-3.5-flash-lite", "gemini-flash-lite-latest"]  # dicoba berurutan kalau model utama sibuk (503/429)
TEMPERATURE = 0.4                   # rendah supaya pencatatan konsisten, tapi masih luwes saat ngobrol
THINKING_LEVEL = "low"              # "minimal", "low", "medium", "high", atau None; makin rendah makin cepat
MAX_TOOL_CALLS = 12                 # batas function call dalam satu pesan
RETRY_PERCOBAAN = 3                 # percobaan ulang otomatis per request saat Gemini sibuk
GAYA_BAHASA = "santai"              # "santai" atau "formal"
MODE_SARAN = "suportif"             # "suportif" atau "tegas"
ZONA_WAKTU = "Asia/Jakarta"         # dipakai untuk menentukan "hari ini" dan "kemarin"
RIWAYAT_CHAT_DIMUAT = 20            # jumlah pesan terakhir yang dimuat ulang saat aplikasi dibuka lagi
DATA_DIR = os.environ.get("DOMPY_DATA_DIR", "data")  # lokasi database

### API key Gemini

Ambil API key gratis di [Google AI Studio](https://aistudio.google.com/apikey). Sel berikut membacanya dengan urutan:
1. **Colab**: menu 🔑 *Secrets* di sidebar kiri. Buat secret bernama `GEMINI_API_KEY` dan aktifkan *Notebook access*.
2. **Lokal**: environment variable `GEMINI_API_KEY`, atau file `.env` di folder yang sama berisi `GEMINI_API_KEY=...`.
3. Kalau semuanya kosong, notebook meminta key lewat input tersembunyi.

Key hanya disimpan di environment variable sesi ini (tidak ditulis ke file), lalu diwariskan ke proses Streamlit.

Set `SIMPAN_DI_DRIVE = True` kalau kamu di Colab dan ingin database disimpan di Google Drive supaya tidak hilang saat runtime direset.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
SIMPAN_DI_DRIVE = False


def load_api_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
            if key:
                return key
        except Exception:
            pass
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    if os.path.exists(".env"):
        with open(".env", encoding="utf-8") as f:
            for line in f:
                if line.strip().startswith("GEMINI_API_KEY="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    from getpass import getpass
    return getpass("Masukkan GEMINI_API_KEY: ")


os.environ["GEMINI_API_KEY"] = load_api_key()

if IN_COLAB and SIMPAN_DI_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DOMPY_DATA_DIR"] = "/content/drive/MyDrive/Dompy"

print("API key :", "terbaca ✅" if os.environ["GEMINI_API_KEY"] else "belum ada ❌")
print("Data    :", os.environ.get("DOMPY_DATA_DIR", "data"))

## 3. Database dan fungsi bantu (`database.py`)

Tiga tabel SQLite:
- `budgets`: budget per kategori beserta rentang tanggalnya (harian, mingguan, bulanan, atau custom).
- `expenses`: setiap pengeluaran yang dicatat.
- `chat_log`: riwayat obrolan, supaya percakapan bisa dilanjutkan setelah aplikasi dibuka ulang.

Fungsi `status_budget` menghitung semua angka yang nanti dilaporkan Dompy: terpakai, sisa, jatah per hari, proyeksi akhir periode, dan level peringatan.

In [ ]:
%%writefile database.py
import calendar
import os
import sqlite3
from contextlib import contextmanager
from datetime import date, datetime, timedelta, timezone

import config

try:
    from zoneinfo import ZoneInfo
    TZ = ZoneInfo(config.ZONA_WAKTU)
except Exception:
    TZ = timezone(timedelta(hours=7))  # cadangan kalau data zona waktu tidak tersedia

os.makedirs(config.DATA_DIR, exist_ok=True)
DB_PATH = os.path.join(config.DATA_DIR, "dompy.db")

KATEGORI = ["Makanan", "Transportasi", "Belanja", "Hiburan", "Tagihan", "Kesehatan", "Pendidikan", "Lainnya"]
SINONIM_KATEGORI = {
    "makan": "Makanan", "minum": "Makanan", "jajan": "Makanan", "kuliner": "Makanan", "food": "Makanan",
    "transport": "Transportasi", "bensin": "Transportasi", "ojol": "Transportasi", "parkir": "Transportasi",
    "shopping": "Belanja", "groceries": "Belanja",
    "nonton": "Hiburan", "game": "Hiburan", "langganan": "Hiburan",
    "listrik": "Tagihan", "pulsa": "Tagihan", "internet": "Tagihan", "kos": "Tagihan", "sewa": "Tagihan",
    "obat": "Kesehatan", "dokter": "Kesehatan",
    "buku": "Pendidikan", "kursus": "Pendidikan", "kuliah": "Pendidikan",
    "lain": "Lainnya", "lain-lain": "Lainnya",
    "total": "Total", "semua": "Total", "keseluruhan": "Total", "all": "Total",
}
NAMA_HARI = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]


@contextmanager
def db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()


def init_db():
    with db() as conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS budgets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT NOT NULL,
            amount INTEGER NOT NULL,
            period TEXT NOT NULL,
            start_date TEXT NOT NULL,
            end_date TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS expenses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT NOT NULL,
            amount INTEGER NOT NULL,
            description TEXT,
            date TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS chat_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
        """)


init_db()


def normalisasi_kategori(nama):
    key = (nama or "").strip().lower()
    if not key:
        return "Lainnya"
    for k in KATEGORI:
        if key == k.lower():
            return k
    return SINONIM_KATEGORI.get(key, nama.strip().title())


def rupiah(x):
    x = int(round(x))
    return ("-" if x < 0 else "") + "Rp" + f"{abs(x):,}".replace(",", ".")


def hari_ini():
    return datetime.now(TZ).date()


def sekarang():
    return datetime.now(TZ).isoformat(timespec="seconds")


def ke_tanggal(teks):
    teks = (teks or "").strip()
    return date.fromisoformat(teks) if teks else hari_ini()


def akhir_bulan(d):
    return d.replace(day=calendar.monthrange(d.year, d.month)[1])


def tambah_bulan(d, n=1):
    bulan = d.month - 1 + n
    tahun = d.year + bulan // 12
    bulan = bulan % 12 + 1
    return date(tahun, bulan, min(d.day, calendar.monthrange(tahun, bulan)[1]))


def hitung_periode(period, start_date="", end_date=""):
    """Mengubah nama periode menjadi (nama_baku, tanggal_mulai, tanggal_selesai)."""
    p = (period or "").strip().lower()
    today = hari_ini()
    if p in ("harian", "hari", "daily", "hari ini"):
        mulai = ke_tanggal(start_date)
        return "harian", mulai, mulai
    if p in ("mingguan", "minggu", "weekly", "minggu ini"):
        mulai = ke_tanggal(start_date) if start_date else today - timedelta(days=today.weekday())
        return "mingguan", mulai, mulai + timedelta(days=6)
    if p in ("bulanan", "bulan", "monthly", "bulan ini"):
        if start_date:
            mulai = ke_tanggal(start_date)
            return "bulanan", mulai, tambah_bulan(mulai) - timedelta(days=1)
        return "bulanan", today.replace(day=1), akhir_bulan(today)
    if p == "custom":
        if not (start_date and end_date):
            raise ValueError("Periode custom butuh start_date dan end_date (format YYYY-MM-DD).")
        mulai, selesai = ke_tanggal(start_date), ke_tanggal(end_date)
        if selesai < mulai:
            raise ValueError("end_date tidak boleh sebelum start_date.")
        return "custom", mulai, selesai
    raise ValueError(f"Periode '{period}' tidak dikenal. Pilih harian, mingguan, bulanan, atau custom.")


def periode_sebelumnya(mulai, selesai):
    """Rentang dengan panjang sama tepat sebelum periode ini (bulan kalender tetap dihitung per bulan)."""
    if mulai.day == 1 and selesai == akhir_bulan(mulai):
        prev_selesai = mulai - timedelta(days=1)
        return prev_selesai.replace(day=1), prev_selesai
    panjang = (selesai - mulai).days + 1
    return mulai - timedelta(days=panjang), mulai - timedelta(days=1)


def total_pengeluaran(mulai, selesai, kategori=None):
    sql = "SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE date BETWEEN ? AND ?"
    args = [mulai.isoformat(), selesai.isoformat()]
    if kategori and kategori != "Total":
        sql += " AND category = ?"
        args.append(kategori)
    with db() as conn:
        return conn.execute(sql, args).fetchone()[0]


def ambil_pengeluaran(mulai, selesai):
    with db() as conn:
        rows = conn.execute(
            "SELECT * FROM expenses WHERE date BETWEEN ? AND ? ORDER BY date, id",
            (mulai.isoformat(), selesai.isoformat()),
        ).fetchall()
    return [dict(r) for r in rows]


def budget_aktif(pada=None):
    """Budget yang berlaku pada tanggal tertentu, satu per kategori (yang terbaru dibuat)."""
    pada = (pada or hari_ini()).isoformat()
    with db() as conn:
        rows = conn.execute(
            "SELECT * FROM budgets WHERE start_date <= ? AND end_date >= ? ORDER BY created_at DESC, id DESC",
            (pada, pada),
        ).fetchall()
    hasil = {}
    for r in rows:
        hasil.setdefault(r["category"], dict(r))
    return list(hasil.values())


def status_budget(b):
    """Kondisi satu budget: terpakai, sisa, jatah per hari, proyeksi, dan level peringatan."""
    mulai, selesai = date.fromisoformat(b["start_date"]), date.fromisoformat(b["end_date"])
    terpakai = total_pengeluaran(mulai, selesai, b["category"])
    sisa = b["amount"] - terpakai
    persen = terpakai / b["amount"] * 100 if b["amount"] else 0
    today = hari_ini()
    total_hari = (selesai - mulai).days + 1
    hari_berjalan = max(0, min((today - mulai).days + 1, total_hari))
    hari_tersisa = (selesai - max(today, mulai)).days + 1 if today <= selesai else 0
    proyeksi = terpakai / hari_berjalan * total_hari if hari_berjalan else terpakai
    level = "lewat budget" if persen >= 100 else "waspada" if persen >= 75 else "aman"
    return {
        "kategori": b["category"],
        "periode": b["period"],
        "mulai": b["start_date"],
        "selesai": b["end_date"],
        "budget": rupiah(b["amount"]),
        "terpakai": rupiah(terpakai),
        "sisa": rupiah(sisa),
        "persen_terpakai": round(persen, 1),
        "hari_tersisa": hari_tersisa,
        "jatah_per_hari": rupiah(sisa / hari_tersisa) if hari_tersisa and sisa > 0 else rupiah(0),
        "proyeksi_akhir_periode": rupiah(proyeksi),
        "level": level,
    }


def jejak_data():
    """Sidik jari isi tabel budget & transaksi, untuk tahu apakah ada yang berubah."""
    with db() as conn:
        return tuple(conn.execute(
            "SELECT (SELECT COUNT(*) FROM expenses), (SELECT COALESCE(MAX(id), 0) FROM expenses),"
            " (SELECT COUNT(*) FROM budgets), (SELECT COALESCE(MAX(id), 0) FROM budgets)").fetchone())


def simpan_chat(role, content):
    with db() as conn:
        conn.execute("INSERT INTO chat_log (role, content, created_at) VALUES (?, ?, ?)", (role, content, sekarang()))


def muat_chat(limit=config.RIWAYAT_CHAT_DIMUAT):
    with db() as conn:
        rows = conn.execute("SELECT role, content FROM chat_log ORDER BY id DESC LIMIT ?", (limit,)).fetchall()
    rows = [dict(r) for r in reversed(rows)]
    while rows and rows[0]["role"] != "user":  # riwayat Gemini harus diawali pesan user
        rows.pop(0)
    return rows


def hapus_chat():
    with db() as conn:
        conn.execute("DELETE FROM chat_log")

## 4. Tools untuk Gemini (`tools.py`)

Fungsi-fungsi di bawah adalah "alat" Dompy. Gemini membaca *docstring* dan tipe parameternya, lalu memutuskan sendiri kapan harus memanggil fungsi yang mana.

| Tool | Kegunaan |
|---|---|
| `atur_budget` | Menyimpan budget per kategori untuk periode harian/mingguan/bulanan/custom |
| `catat_pengeluaran` | Mencatat pengeluaran lalu mengembalikan sisa budget terkait |
| `cek_budget` | Melihat kondisi semua budget yang sedang aktif |
| `ringkasan_pengeluaran` | Rekap pengeluaran pada rentang tanggal tertentu |
| `daftar_transaksi` | Daftar transaksi terakhir (beserta ID untuk koreksi) |
| `hapus_transaksi` | Menghapus transaksi yang salah catat |
| `analisis_pengeluaran` | Data evaluasi periode: perbandingan dengan periode lalu, kebiasaan belanja, dan usulan budget |

In [ ]:
%%writefile tools.py
from collections import Counter, defaultdict
from datetime import date

from database import (NAMA_HARI, ambil_pengeluaran, budget_aktif, db, hari_ini, hitung_periode, ke_tanggal,
                      normalisasi_kategori, periode_sebelumnya, rupiah, sekarang, status_budget)


def atur_budget(category: str, amount: int, period: str, start_date: str = "", end_date: str = "") -> dict:
    """Menyimpan atau mengganti budget untuk satu kategori pengeluaran.

    Args:
        category: Kategori, misalnya Makanan, Transportasi, Belanja, Hiburan, Tagihan, Kesehatan, Pendidikan, Lainnya. Gunakan "Total" untuk budget seluruh pengeluaran.
        amount: Nominal budget dalam rupiah sebagai angka bulat, contoh 300000.
        period: Salah satu dari "harian", "mingguan", "bulanan", atau "custom".
        start_date: Tanggal mulai (YYYY-MM-DD). Kosongkan untuk periode berjalan (hari ini, minggu ini Senin-Minggu, atau bulan ini).
        end_date: Tanggal selesai (YYYY-MM-DD). Hanya diisi untuk period "custom".
    """
    try:
        amount = int(round(float(amount)))
        if amount <= 0:
            return {"status": "gagal", "pesan": "Nominal budget harus lebih dari 0."}
        kategori = normalisasi_kategori(category)
        periode, mulai, selesai = hitung_periode(period, start_date, end_date)
    except ValueError as e:
        return {"status": "gagal", "pesan": str(e)}

    with db() as conn:
        diganti = conn.execute(
            "DELETE FROM budgets WHERE category = ? AND start_date <= ? AND end_date >= ?",
            (kategori, selesai.isoformat(), mulai.isoformat()),
        ).rowcount
        cur = conn.execute(
            "INSERT INTO budgets (category, amount, period, start_date, end_date, created_at) VALUES (?, ?, ?, ?, ?, ?)",
            (kategori, amount, periode, mulai.isoformat(), selesai.isoformat(), sekarang()),
        )
        row = conn.execute("SELECT * FROM budgets WHERE id = ?", (cur.lastrowid,)).fetchone()
    hasil = status_budget(dict(row))
    hasil["status"] = "ok"
    hasil["mengganti_budget_lama"] = diganti > 0
    return hasil


def catat_pengeluaran(category: str, amount: int, description: str, expense_date: str = "") -> dict:
    """Mencatat satu pengeluaran dan mengembalikan sisa budget yang terkait. Panggil sekali untuk setiap item.

    Args:
        category: Kategori pengeluaran, misalnya Makanan, Transportasi, Belanja, Hiburan, Tagihan, Kesehatan, Pendidikan, Lainnya.
        amount: Nominal dalam rupiah sebagai angka bulat, contoh 18000.
        description: Keterangan singkat, contoh "bakso" atau "isi bensin".
        expense_date: Tanggal transaksi (YYYY-MM-DD). Kosongkan jika hari ini.
    """
    try:
        amount = int(round(float(amount)))
        if amount <= 0:
            return {"status": "gagal", "pesan": "Nominal pengeluaran harus lebih dari 0."}
        kategori = normalisasi_kategori(category)
        if kategori == "Total":
            kategori = "Lainnya"
        tanggal = ke_tanggal(expense_date)
    except ValueError as e:
        return {"status": "gagal", "pesan": str(e)}

    with db() as conn:
        cur = conn.execute(
            "INSERT INTO expenses (category, amount, description, date, created_at) VALUES (?, ?, ?, ?, ?)",
            (kategori, amount, (description or "").strip(), tanggal.isoformat(), sekarang()),
        )
        expense_id = cur.lastrowid

    terkait = [status_budget(b) for b in budget_aktif(tanggal) if b["category"] in (kategori, "Total")]
    return {
        "status": "ok",
        "id": expense_id,
        "kategori": kategori,
        "jumlah": rupiah(amount),
        "deskripsi": description,
        "tanggal": tanggal.isoformat(),
        "budget_terkait": terkait,
        "catatan": "" if terkait else f"Belum ada budget untuk {kategori} pada tanggal ini.",
    }


def cek_budget(category: str = "") -> dict:
    """Melihat kondisi budget yang sedang aktif hari ini: terpakai, sisa, jatah per hari, dan proyeksi.

    Args:
        category: Nama kategori tertentu. Kosongkan untuk melihat semua budget aktif.
    """
    semua = budget_aktif()
    if category:
        kategori = normalisasi_kategori(category)
        semua = [b for b in semua if b["category"] == kategori]
    if not semua:
        return {"status": "kosong", "pesan": "Belum ada budget aktif" + (f" untuk {category}." if category else ".")}
    return {"status": "ok", "tanggal_hari_ini": hari_ini().isoformat(), "budget": [status_budget(b) for b in semua]}


def ringkasan_pengeluaran(start_date: str = "", end_date: str = "") -> dict:
    """Rekap pengeluaran pada rentang tanggal: total, per kategori, rata-rata harian, dan transaksi terbesar.

    Args:
        start_date: Tanggal awal (YYYY-MM-DD). Kosongkan untuk awal bulan ini.
        end_date: Tanggal akhir (YYYY-MM-DD). Kosongkan untuk hari ini.
    """
    try:
        mulai = ke_tanggal(start_date) if start_date else hari_ini().replace(day=1)
        selesai = ke_tanggal(end_date)
    except ValueError as e:
        return {"status": "gagal", "pesan": str(e)}
    data = ambil_pengeluaran(mulai, selesai)
    if not data:
        return {"status": "kosong", "pesan": f"Tidak ada pengeluaran antara {mulai} dan {selesai}."}
    total = sum(d["amount"] for d in data)
    per_kategori = defaultdict(int)
    for d in data:
        per_kategori[d["category"]] += d["amount"]
    terbesar = max(data, key=lambda d: d["amount"])
    return {
        "status": "ok",
        "rentang": f"{mulai} s/d {selesai}",
        "total": rupiah(total),
        "jumlah_transaksi": len(data),
        "rata_rata_per_hari": rupiah(total / ((selesai - mulai).days + 1)),
        "per_kategori": {k: f"{rupiah(v)} ({v / total * 100:.0f}%)"
                         for k, v in sorted(per_kategori.items(), key=lambda x: -x[1])},
        "transaksi_terbesar": f"{terbesar['description']} ({terbesar['category']}) {rupiah(terbesar['amount'])} pada {terbesar['date']}",
    }


def daftar_transaksi(limit: int = 10, category: str = "") -> dict:
    """Menampilkan transaksi terakhir beserta ID-nya. Dipakai juga untuk mencari transaksi yang mau dikoreksi.

    Args:
        limit: Jumlah transaksi yang ditampilkan, default 10.
        category: Filter kategori. Kosongkan untuk semua kategori.
    """
    sql, args = "SELECT * FROM expenses", []
    if category:
        sql += " WHERE category = ?"
        args.append(normalisasi_kategori(category))
    sql += " ORDER BY date DESC, id DESC LIMIT ?"
    args.append(max(1, min(int(limit), 50)))
    with db() as conn:
        rows = conn.execute(sql, args).fetchall()
    return {"status": "ok", "transaksi": [
        {"id": r["id"], "tanggal": r["date"], "kategori": r["category"],
         "deskripsi": r["description"], "jumlah": rupiah(r["amount"])} for r in rows]}


def hapus_transaksi(expense_id: int) -> dict:
    """Menghapus satu transaksi berdasarkan ID (misalnya karena salah catat).

    Args:
        expense_id: ID transaksi dari daftar_transaksi atau dari hasil catat_pengeluaran.
    """
    with db() as conn:
        row = conn.execute("SELECT * FROM expenses WHERE id = ?", (int(expense_id),)).fetchone()
        if not row:
            return {"status": "gagal", "pesan": f"Transaksi dengan ID {expense_id} tidak ditemukan."}
        conn.execute("DELETE FROM expenses WHERE id = ?", (int(expense_id),))
    return {"status": "ok", "dihapus": f"{row['description']} ({row['category']}) {rupiah(row['amount'])} pada {row['date']}"}


def analisis_pengeluaran(period: str = "bulanan") -> dict:
    """Mengumpulkan data evaluasi periode berjalan sebagai bahan masukan dan rekomendasi.

    Hasilnya berisi perbandingan dengan periode sebelumnya per kategori, kepatuhan terhadap budget,
    item yang paling sering dibeli, hari paling boros, pola weekday vs weekend, serta rata-rata 3 periode
    terakhir per kategori yang bisa dipakai untuk mengusulkan budget periode berikutnya.

    Args:
        period: "mingguan" atau "bulanan".
    """
    try:
        nama, mulai, selesai = hitung_periode(period)
    except ValueError as e:
        return {"status": "gagal", "pesan": str(e)}
    sampai = min(selesai, hari_ini())
    data = ambil_pengeluaran(mulai, sampai)
    prev_mulai, prev_selesai = periode_sebelumnya(mulai, selesai)

    def per_kat(rows):
        hasil = defaultdict(int)
        for r in rows:
            hasil[r["category"]] += r["amount"]
        return hasil

    now_k, prev_k = per_kat(data), per_kat(ambil_pengeluaran(prev_mulai, prev_selesai))
    perbandingan = {}
    for k in sorted(set(now_k) | set(prev_k), key=lambda k: -now_k.get(k, 0)):
        a, b = now_k.get(k, 0), prev_k.get(k, 0)
        perbandingan[k] = {"periode_ini": rupiah(a), "periode_lalu": rupiah(b),
                           "perubahan": f"{(a - b) / b * 100:+.0f}%" if b else "baru"}

    # rata-rata 3 periode penuh sebelumnya, bahan usulan budget
    riwayat = defaultdict(list)
    m, s = mulai, selesai
    for _ in range(3):
        m, s = periode_sebelumnya(m, s)
        kat = per_kat(ambil_pengeluaran(m, s))
        for k in set(kat) | set(now_k):
            riwayat[k].append(kat.get(k, 0))
    rata3 = {k: rupiah(sum(v) / len(v)) for k, v in riwayat.items() if sum(v) > 0}

    item = Counter((r["description"] or "").strip().lower() for r in data if r["description"])
    per_hari = defaultdict(int)
    for r in data:
        per_hari[r["date"]] += r["amount"]
    weekday = [v for d, v in per_hari.items() if date.fromisoformat(d).weekday() < 5]
    weekend = [v for d, v in per_hari.items() if date.fromisoformat(d).weekday() >= 5]
    terboros = max(per_hari.items(), key=lambda x: x[1]) if per_hari else None

    total = sum(now_k.values())
    hari_berjalan = (sampai - mulai).days + 1
    return {
        "status": "ok" if data else "kosong",
        "periode": nama,
        "rentang": f"{mulai} s/d {selesai} (data sampai {sampai})",
        "hari_berjalan": hari_berjalan,
        "total_periode_ini": rupiah(total),
        "total_periode_lalu": rupiah(sum(prev_k.values())),
        "rata_rata_per_hari": rupiah(total / hari_berjalan),
        "per_kategori": perbandingan,
        "status_budget": [status_budget(b) for b in budget_aktif(sampai)],
        "item_paling_sering": [f"{n} ({c}x)" for n, c in item.most_common(5)],
        "hari_terboros": f"{NAMA_HARI[date.fromisoformat(terboros[0]).weekday()]} {terboros[0]}: {rupiah(terboros[1])}" if terboros else "-",
        "rata_rata_harian_weekday": rupiah(sum(weekday) / len(weekday)) if weekday else rupiah(0),
        "rata_rata_harian_weekend": rupiah(sum(weekend) / len(weekend)) if weekend else rupiah(0),
        "rata_rata_3_periode_sebelumnya": rata3,
    }


TOOLS = [atur_budget, catat_pengeluaran, cek_budget, ringkasan_pengeluaran,
         daftar_transaksi, hapus_transaksi, analisis_pengeluaran]

## 5. Otak Dompy: Gemini + system prompt (`agent.py`)

*System prompt* dibangun ulang di setiap pesan supaya tanggal "hari ini" selalu benar dan perubahan gaya bahasa dari sidebar langsung berlaku.

Parameter yang bisa dikreasikan:
- **Gaya bahasa**: `santai` (akrab, pakai "kamu", sedikit emoji) atau `formal` (sopan, pakai "Anda", tanpa emoji).
- **Mode saran**: `suportif` (menyemangati) atau `tegas` (lugas soal pemborosan).
- **Temperature**: rendah (0.4) agar interpretasi nominal dan kategori konsisten.

**Kalau Gemini sibuk (503).** Model gratisan sering penuh. Client diberi `HttpRetryOptions`, jadi request yang gagal dengan kode 429/500/503/504 diulang otomatis dengan jeda yang makin panjang. Kalau masih gagal juga, pesan dikirim ke model berikutnya di `MODEL_CADANGAN` bersama riwayat chat. Ada satu pengecualian: kalau sebelum gagal sudah ada tool yang sempat menulis ke database, pesan tidak dikirim ulang supaya transaksi tidak tercatat dua kali.

`THINKING_LEVEL = "low"` mempercepat jawaban. Mencatat pengeluaran tidak butuh model berpikir panjang.

SDK `google-genai` menjalankan *automatic function calling*: kalau Gemini memutuskan memanggil tool, SDK langsung mengeksekusi fungsi Python-nya, mengirim hasilnya kembali ke Gemini, dan baru mengembalikan jawaban akhir.

In [ ]:
%%writefile agent.py
import os

from google import genai
from google.genai import errors, types

import config
from database import KATEGORI, NAMA_HARI, hari_ini, jejak_data, muat_chat, simpan_chat
from tools import TOOLS

GAYA = {
    "santai": "Bahasa santai dan akrab seperti teman dekat. Panggil user 'kamu'. Boleh pakai maksimal 2 emoji per jawaban.",
    "formal": "Bahasa Indonesia baku, sopan, dan profesional. Panggil user 'Anda'. Jangan pakai emoji maupun slang.",
}
MODE = {
    "suportif": "Saat memberi masukan, apresiasi hal yang sudah baik dulu, lalu sampaikan koreksi dengan lembut dan memotivasi.",
    "tegas": "Saat memberi masukan, langsung tunjuk pemborosan dengan lugas dan beri target yang jelas. Tetap sopan, tanpa basa-basi.",
}
KODE_SIBUK = (429, 500, 503, 504)  # error sementara: model penuh, kena limit, atau server lagi gangguan


def load_api_key():
    key = os.environ.get("GEMINI_API_KEY")
    if not key and os.path.exists(".env"):
        with open(".env", encoding="utf-8") as f:
            for line in f:
                if line.strip().startswith("GEMINI_API_KEY="):
                    key = line.split("=", 1)[1].strip().strip('"').strip("'")
    return key


def buat_client():
    key = load_api_key()
    if not key:
        raise RuntimeError("GEMINI_API_KEY belum diatur. Isi di Colab Secrets, environment variable, atau file .env.")
    # SDK mengulang sendiri request yang gagal karena server sibuk, dengan jeda yang makin lama (1 s, 2 s, ...)
    retry = types.HttpRetryOptions(attempts=config.RETRY_PERCOBAAN, initial_delay=1, max_delay=8,
                                   http_status_codes=list(KODE_SIBUK))
    return genai.Client(api_key=key, http_options=types.HttpOptions(retry_options=retry))


def buat_system_prompt(gaya, mode):
    today = hari_ini()
    return f"""Kamu adalah Dompy, asisten pengelola budget pribadi berbahasa Indonesia.
Hari ini {NAMA_HARI[today.weekday()]}, {today.isoformat()} (WIB).

ATURAN KERJA
1. Semua pencatatan dan perhitungan WAJIB lewat tools. Jangan pernah menghitung sisa budget atau total sendiri; pakai angka dari hasil tools.
2. Ubah nominal informal ke angka bulat: "18rb"/"18k" = 18000, "1,5jt" = 1500000, "gopek" = 500, "goceng" = 5000, "ceban" = 10000, "gocap" = 50000, "cepek" = 100000.
3. Ubah tanggal relatif (kemarin, Sabtu lalu, tanggal 3) ke format YYYY-MM-DD berdasarkan tanggal hari ini.
4. Satu pesan bisa berisi beberapa pengeluaran: panggil catat_pengeluaran untuk setiap item.
5. Kategori yang tersedia: {", ".join(KATEGORI)}. Pakai "Total" untuk budget keseluruhan. Kalau ragu, pilih yang paling masuk akal dan sebutkan kategorinya.
6. Jika user menyebut budget tanpa periode yang jelas, tanyakan dulu: harian, mingguan, bulanan, atau rentang tanggal tertentu.
7. Setelah mencatat pengeluaran, sebutkan sisa budget kategori itu dan jatah per hari jika ada budgetnya. Beri peringatan jika level "waspada" atau "lewat budget". Jika belum ada budget, tawarkan untuk membuatnya.
8. Jika diminta evaluasi, masukan, atau rekap periode, panggil analisis_pengeluaran lalu berikan: ringkasan singkat, 2-4 saran konkret berdasarkan data (sebut item/kategori nyata), dan usulan budget periode berikutnya.
9. Untuk koreksi ("eh salah, harusnya 15rb"), cari ID lewat daftar_transaksi, hapus_transaksi, lalu catat ulang.
10. Fokus pada keuangan pribadi. Topik lain dijawab singkat lalu arahkan kembali. Jangan merekomendasikan saham, kripto, atau produk investasi tertentu.

FORMAT
- Tulis rupiah seperti Rp25.000.
- Jawaban ringkas (maksimal sekitar 6 baris) kecuali user minta detail atau sedang evaluasi.
- Boleh pakai bullet list untuk rekap.

GAYA BAHASA: {GAYA.get(gaya, GAYA["santai"])}
MODE SARAN: {MODE.get(mode, MODE["suportif"])}
"""


def buat_config(gaya=config.GAYA_BAHASA, mode=config.MODE_SARAN, thinking=True):
    return types.GenerateContentConfig(
        system_instruction=buat_system_prompt(gaya, mode),
        tools=TOOLS,
        temperature=config.TEMPERATURE,
        thinking_config=types.ThinkingConfig(thinking_level=config.THINKING_LEVEL)
        if thinking and config.THINKING_LEVEL else None,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=config.MAX_TOOL_CALLS),
    )


def buat_sesi_chat(client, model=config.MODEL_NAME, history=None):
    """Sesi chat baru. Tanpa `history`, riwayat obrolan terakhir dimuat dari database."""
    if history is None:
        history = [types.Content(role="user" if r["role"] == "user" else "model", parts=[types.Part(text=r["content"])])
                   for r in muat_chat()]
    return client.chats.create(model=model, config=buat_config(), history=history)


def tools_terpakai(response):
    """Daftar function call pada giliran terakhir, untuk ditampilkan di UI."""
    riwayat = response.automatic_function_calling_history or []
    awal = 0
    for i, content in enumerate(riwayat):  # cari pesan user (teks) terakhir
        if content.role == "user" and any(p.text for p in content.parts or []):
            awal = i
    calls = []
    for content in riwayat[awal:]:
        for p in content.parts or []:
            if p.function_call:
                args = ", ".join(f"{k}={v!r}" for k, v in (p.function_call.args or {}).items())
                calls.append(f"{p.function_call.name}({args})")
    return calls


def pesan_error(e):
    """Ubah error Gemini jadi pesan yang bisa dipahami user."""
    kode = getattr(e, "code", None)
    if kode == 429:
        return "Kuota Gemini gratisan lagi habis/terlalu sering dipakai. Tunggu sekitar 1 menit lalu kirim ulang ya."
    if kode in KODE_SIBUK:
        return "Server Gemini lagi penuh (semua model yang dicoba sedang sibuk). Coba kirim ulang sebentar lagi ya."
    if kode in (401, 403) or "API key" in str(e):
        return "API key Gemini ditolak. Cek lagi isi `GEMINI_API_KEY`."
    return f"Gagal menghubungi Gemini: {e}"


def tanya_dompy(client, sesi, pesan, gaya=config.GAYA_BAHASA, mode=config.MODE_SARAN):
    """Kirim pesan ke Dompy.

    Kalau model utama sibuk, pesan dikirim ulang ke model cadangan (riwayat obrolan ikut dibawa).
    Mengembalikan (jawaban, daftar_tool_yang_dipanggil, sesi, model_yang_menjawab). `sesi` bisa berganti
    ke model cadangan, jadi simpan lagi nilai kembaliannya.
    """
    model_dicoba = [sesi._model] + [m for m in config.MODEL_CADANGAN if m != sesi._model]
    error_terakhir = None
    for model in model_dicoba:
        if model != sesi._model:
            sesi = buat_sesi_chat(client, model, history=sesi.get_history(curated=True))
        sebelum = jejak_data()
        for thinking in (True, False):
            try:
                response = sesi.send_message(pesan, config=buat_config(gaya, mode, thinking))
                jawaban = (response.text or "").strip() or "Maaf, aku belum bisa menjawab itu. Coba ulangi dengan kalimat lain ya."
                simpan_chat("user", pesan)
                simpan_chat("assistant", jawaban)
                return jawaban, tools_terpakai(response), sesi, model
            except errors.APIError as e:
                error_terakhir = e
                if e.code == 400 and "thinking" in str(e).lower() and thinking:
                    continue  # model ini tidak mendukung thinking_level, ulangi tanpa itu
                break
            except Exception as e:  # koneksi putus, dsb.
                return f"⚠️ Gagal menghubungi Gemini: {e}", [], sesi, None
        if jejak_data() != sebelum:
            # Sebagian tool sudah sempat jalan. Jangan kirim ulang supaya transaksi tidak tercatat dobel.
            return ("⚠️ Gemini terputus di tengah jalan, tapi sebagian data sudah sempat tersimpan. "
                    "Cek tab Transaksi dulu sebelum mengirim ulang ya."), [], sesi, None
        if getattr(error_terakhir, "code", None) not in KODE_SIBUK + (404,):
            break  # error bukan karena model sibuk/tidak ada, ganti model tidak akan membantu
    return f"⚠️ {pesan_error(error_terakhir)}", [], sesi, None

## 6. Grafik dashboard (`charts.py`)

In [ ]:
%%writefile charts.py
from datetime import timedelta

import pandas as pd
import plotly.graph_objects as go

from database import ambil_pengeluaran, hari_ini, rupiah

WARNA_KATEGORI = {
    "Makanan": "#f97316", "Transportasi": "#3b82f6", "Belanja": "#a855f7", "Hiburan": "#ec4899",
    "Tagihan": "#64748b", "Kesehatan": "#14b8a6", "Pendidikan": "#eab308", "Lainnya": "#94a3b8",
}
FONT = "Plus Jakarta Sans, sans-serif"
TEKS_PUDAR = "#6b7570"
GAYA_DASAR = dict(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)", margin=dict(l=4, r=4, t=8, b=4),
                  font=dict(family=FONT, color="#17201b", size=12),
                  hoverlabel=dict(bgcolor="#17201b", font=dict(family=FONT, color="#ffffff"), bordercolor="#17201b"))


def figur_kosong(pesan, tinggi=260):
    fig = go.Figure()
    fig.add_annotation(text=pesan, showarrow=False, font=dict(size=13, color=TEKS_PUDAR))
    fig.update_layout(xaxis_visible=False, yaxis_visible=False, height=tinggi, **GAYA_DASAR)
    return fig


def grafik_kategori():
    today = hari_ini()
    data = ambil_pengeluaran(today.replace(day=1), today)
    if not data:
        return figur_kosong("Belum ada pengeluaran bulan ini.")
    df = pd.DataFrame(data).groupby("category", as_index=False)["amount"].sum().sort_values("amount", ascending=False)
    fig = go.Figure(go.Pie(labels=df["category"], values=df["amount"], hole=0.62, sort=False, direction="clockwise",
                           marker=dict(colors=[WARNA_KATEGORI.get(k, "#94a3b8") for k in df["category"]],
                                       line=dict(color="#ffffff", width=2)),
                           textinfo="none", customdata=[rupiah(v) for v in df["amount"]],
                           hovertemplate="<b>%{label}</b><br>%{customdata} · %{percent}<extra></extra>"))
    fig.update_layout(height=280, showlegend=True,
                      legend=dict(orientation="v", x=1.02, y=0.5, font=dict(size=12), itemclick=False),
                      annotations=[dict(text=f"<span style='font-size:11px;color:{TEKS_PUDAR}'>Total</span><br>"
                                              f"<b>{rupiah(df['amount'].sum())}</b>",
                                        showarrow=False, font=dict(size=15))],
                      **GAYA_DASAR)
    return fig


def grafik_harian(hari=30):
    today = hari_ini()
    mulai = today - timedelta(days=hari - 1)
    data = ambil_pengeluaran(mulai, today)
    if not data:
        return figur_kosong(f"Belum ada pengeluaran {hari} hari terakhir.")
    pivot = pd.DataFrame(data).pivot_table(index="date", columns="category", values="amount", aggfunc="sum", fill_value=0)
    pivot = pivot.reindex([(mulai + timedelta(days=i)).isoformat() for i in range(hari)], fill_value=0)
    tanggal = pd.to_datetime(pivot.index)
    fig = go.Figure()
    for kat in pivot.columns:
        fig.add_bar(x=tanggal, y=pivot[kat], name=kat, marker_color=WARNA_KATEGORI.get(kat, "#94a3b8"),
                    customdata=[rupiah(v) for v in pivot[kat]],
                    hovertemplate=f"{kat}: %{{customdata}}<extra></extra>")
    fig.update_layout(barmode="stack", height=280, bargap=0.25, hovermode="x unified",
                      legend=dict(orientation="h", y=-0.18, font=dict(size=11)),
                      xaxis=dict(showgrid=False, tickformat="%d %b", tickfont=dict(color=TEKS_PUDAR)),
                      yaxis=dict(gridcolor="#eceee9", tickformat="~s", tickprefix="Rp", tickfont=dict(color=TEKS_PUDAR)),
                      **GAYA_DASAR)
    return fig

## 7. Tampilan Streamlit (`app.py`)

Layout:
- **Header**: sapaan, tanggal, dan tiga kartu ringkasan (hari ini, minggu ini, bulan ini) lengkap dengan perbandingan ke periode sebelumnya.
- **Sidebar**: pengaturan gaya bahasa dan mode saran, tombol mulai obrolan baru, dan info model yang dipakai.
- **Kolom kiri**: chat dengan Dompy plus chip contoh pesan yang bisa diklik. Jawaban yang memakai tool punya expander "Detail" berisi function call yang dijalankan Gemini.
- **Kolom kanan**: kartu progress budget (hijau/kuning/merah), grafik per kategori dan 30 hari terakhir, serta tabel transaksi. Setelah pesan diproses halaman digambar ulang, jadi angkanya selalu terbaru.

Tema (warna, font Plus Jakarta Sans, sudut membulat) diatur di `.streamlit/config.toml`, sisanya lewat sedikit CSS di awal `app.py`.

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#0f9d6e"
backgroundColor = "#f5f6f3"
secondaryBackgroundColor = "#eceee9"
textColor = "#17201b"
borderColor = "#e2e5df"
font = "Plus Jakarta Sans:https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap"
baseRadius = "12px"
buttonRadius = "full"
showWidgetBorder = true
chartCategoricalColors = ["#f97316", "#3b82f6", "#a855f7", "#ec4899", "#64748b", "#14b8a6", "#eab308", "#94a3b8"]

[theme.sidebar]
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f5f6f3"

[server]
headless = true

[client]
toolbarMode = "minimal"

[browser]
gatherUsageStats = false

In [ ]:
%%writefile app.py
from datetime import timedelta
from html import escape

import pandas as pd
import streamlit as st

import agent
import charts
import config
from database import (NAMA_HARI, budget_aktif, hapus_chat, hari_ini, muat_chat, rupiah, status_budget,
                      tambah_bulan, total_pengeluaran)
from tools import daftar_transaksi

st.set_page_config(page_title="Dompy · Asisten Budget", page_icon="💸", layout="wide")

CONTOH_PESAN = [
    "Budget makan minggu ini 300rb",
    "Makan bakso 18rb + es teh 5rb",
    "Kemarin bensin 30rb, parkir goceng",
    "Sisa budget makanku?",
    "Evaluasi pengeluaran bulan ini",
]
IKON = {"Makanan": "🍜", "Transportasi": "🛵", "Belanja": "🛍️", "Hiburan": "🎬", "Tagihan": "🧾",
        "Kesehatan": "💊", "Pendidikan": "📚", "Lainnya": "📦", "Total": "💰"}
WARNA_LEVEL = {"aman": "#0f9d6e", "waspada": "#f59e0b", "lewat budget": "#e5484d"}
NAMA_BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli", "Agustus", "September", "Oktober",
              "November", "Desember"]

st.html("""<style>
[data-testid="stMainBlockContainer"] {padding-top: 2.2rem; padding-bottom: 2rem; max-width: 1320px;}
[data-testid="stSidebarHeader"] {display: none;}
h1, h2, h3 {letter-spacing: -0.02em;}
.st-key-kartu-chat, .st-key-kartu-budget, .st-key-kartu-grafik {
  background: #fff; border: 1px solid #e2e5df; border-radius: 18px; padding: 1.1rem 1.2rem;
  box-shadow: 0 1px 2px rgba(23,32,27,.04), 0 8px 24px -16px rgba(23,32,27,.12);
}
.judul-kartu {font-weight: 700; font-size: 1rem; margin: 0 0 .15rem 0;}
.sub-kartu {color: #6b7570; font-size: .82rem; margin: 0;}
.sapaan {font-size: 1.85rem; font-weight: 800; letter-spacing: -0.03em; margin: 0; line-height: 1.15;}
.tanggal {color: #6b7570; margin: .25rem 0 0 0; font-size: .95rem;}
.kpi {background: #fff; border: 1px solid #e2e5df; border-radius: 18px; padding: 1rem 1.15rem; height: 100%;}
.kpi.utama {background: linear-gradient(135deg, #0f9d6e 0%, #0b7a56 100%); border: none; color: #fff;}
.kpi .label {font-size: .8rem; font-weight: 600; color: #6b7570; text-transform: uppercase; letter-spacing: .04em;}
.kpi.utama .label, .kpi.utama .delta {color: rgba(255,255,255,.85);}
.kpi .angka {font-size: 1.6rem; font-weight: 800; letter-spacing: -0.03em; margin: .2rem 0 .1rem 0; white-space: nowrap;}
.kpi .delta {font-size: .8rem; color: #6b7570;}
.delta .naik {color: #e5484d; font-weight: 700;} .delta .turun {color: #0f9d6e; font-weight: 700;}
.kpi.utama .delta .naik, .kpi.utama .delta .turun {color: #fff;}
.budget {padding: .8rem 0; border-bottom: 1px solid #eceee9;}
.budget:last-child {border-bottom: none; padding-bottom: 0;}
.budget .atas {display: flex; justify-content: space-between; align-items: baseline; gap: .5rem;}
.budget .nama {font-weight: 700;}
.budget .periode {font-size: .72rem; font-weight: 600; color: #6b7570; background: #f2f4f0;
  padding: .1rem .5rem; border-radius: 999px; margin-left: .35rem; text-transform: capitalize;}
.budget .nominal {font-size: .85rem; color: #6b7570; white-space: nowrap;}
.budget .nominal b {color: #17201b;}
.bar {height: 8px; background: #eceee9; border-radius: 999px; margin: .5rem 0 .4rem 0; overflow: hidden;}
.bar > div {height: 100%; border-radius: 999px;}
.budget .bawah {display: flex; justify-content: space-between; font-size: .78rem; color: #6b7570;}
.kosong {color: #6b7570; font-size: .9rem; padding: 1.2rem 0; text-align: center;}
.logo {display: flex; align-items: center; gap: .6rem; margin: .2rem 0 .3rem 0;}
.logo .ikon {width: 40px; height: 40px; border-radius: 12px; background: #0f9d6e; display: grid;
  place-items: center; font-size: 1.3rem;}
.logo .nama {font-weight: 800; font-size: 1.35rem; letter-spacing: -0.03em;}
.model-info {font-size: .75rem; color: #6b7570;}
[data-testid="stChatMessage"] {background: transparent; padding: .35rem 0;}
.st-key-pilihan-contoh button {font-size: .8rem;}
</style>""")


@st.cache_resource
def get_client():
    return agent.buat_client()


try:
    client = get_client()
except RuntimeError as e:
    st.error(str(e))
    st.stop()

if "sesi" not in st.session_state:
    st.session_state.sesi = agent.buat_sesi_chat(client)
    st.session_state.messages = [{"role": r["role"], "content": r["content"], "tools": []} for r in muat_chat()]


def pilih_contoh():
    st.session_state.kirim = st.session_state["pilihan-contoh"]
    st.session_state["pilihan-contoh"] = None


# ---------- Sidebar ----------
with st.sidebar:
    st.html('<div class="logo"><div class="ikon">💸</div><div class="nama">Dompy</div></div>'
            '<p class="sub-kartu">Catat pengeluaran cukup dengan ngobrol. Gemini yang paham kalimatmu, '
            'Python yang ngitung angkanya.</p>')
    st.divider()
    gaya = st.segmented_control("Gaya bahasa", ["santai", "formal"], default=config.GAYA_BAHASA,
                                format_func=str.capitalize, width="stretch") or config.GAYA_BAHASA
    mode = st.segmented_control("Mode saran", ["suportif", "tegas"], default=config.MODE_SARAN,
                                format_func=str.capitalize, width="stretch") or config.MODE_SARAN
    st.divider()
    if st.button("Mulai obrolan baru", icon=":material/refresh:", width="stretch",
                 help="Riwayat chat dihapus, data budget dan transaksi tetap tersimpan."):
        hapus_chat()
        st.session_state.sesi = agent.buat_sesi_chat(client)
        st.session_state.messages = []
        st.rerun()
    st.html(f'<p class="model-info">Model: <code>{config.MODEL_NAME}</code><br>'
            f'Cadangan: <code>{"</code>, <code>".join(config.MODEL_CADANGAN)}</code></p>')


# ---------- Header + ringkasan ----------
today = hari_ini()


def kartu_kpi(label, sekarang, sebelumnya, pembanding, utama=False):
    if sebelumnya:
        persen = (sekarang - sebelumnya) / sebelumnya * 100
        kelas, panah = ("naik", "▲") if persen > 0 else ("turun", "▼")
        delta = f'<span class="{kelas}">{panah} {abs(persen):.0f}%</span> vs {pembanding}'
    else:
        delta = f"belum ada data {pembanding}"
    return (f'<div class="kpi{" utama" if utama else ""}"><div class="label">{label}</div>'
            f'<div class="angka">{rupiah(sekarang)}</div><div class="delta">{delta}</div></div>')


awal_minggu = today - timedelta(days=today.weekday())
bulan_lalu = tambah_bulan(today, -1)
st.html(f'<p class="sapaan">Halo! Gimana dompetmu hari ini?</p>'
        f'<p class="tanggal">{NAMA_HARI[today.weekday()]}, {today.day} {NAMA_BULAN[today.month - 1]} {today.year}</p>')
k1, k2, k3 = st.columns(3)
k1.html(kartu_kpi("Hari ini", total_pengeluaran(today, today),
                  total_pengeluaran(today - timedelta(days=1), today - timedelta(days=1)), "kemarin"))
k2.html(kartu_kpi("Minggu ini", total_pengeluaran(awal_minggu, today),
                  total_pengeluaran(awal_minggu - timedelta(days=7), today - timedelta(days=7)), "minggu lalu"))
k3.html(kartu_kpi("Bulan ini", total_pengeluaran(today.replace(day=1), today),
                  total_pengeluaran(bulan_lalu.replace(day=1), bulan_lalu), "bulan lalu", utama=True))


def tampilkan_pesan(m):
    with st.chat_message(m["role"], avatar=":material/person:" if m["role"] == "user" else "💸"):
        st.markdown(m["content"])
        if m.get("tools") or m.get("model_cadangan"):
            with st.expander(f"Detail: {len(m.get('tools', []))} tool dipanggil", icon=":material/build:"):
                if m.get("model_cadangan"):
                    st.caption(f"Model utama sibuk, dijawab oleh `{m['model_cadangan']}`.")
                for c in m.get("tools", []):
                    st.code(c, language="python", wrap_lines=True)


# ---------- Chat + dashboard ----------
kiri, kanan = st.columns([1.15, 1], gap="medium")

with kiri:
    with st.container(key="kartu-chat"):
        st.html('<p class="judul-kartu">Ngobrol sama Dompy</p>'
                '<p class="sub-kartu">Tulis budget atau pengeluaran pakai bahasa sehari-hari.</p>')
        kotak_chat = st.container(height=520, border=False)
        st.pills("Contoh", CONTOH_PESAN, key="pilihan-contoh", on_change=pilih_contoh, label_visibility="collapsed")
        prompt = st.chat_input("Contoh: tadi makan nasi padang 22rb") or st.session_state.pop("kirim", None)
        with kotak_chat:
            if not st.session_state.messages and not prompt:
                with st.chat_message("assistant", avatar="💸"):
                    st.markdown("**Halo, aku Dompy!** Mulai dengan kasih tahu budget kamu, lalu ceritakan "
                                "pengeluaranmu kayak lagi chat teman. Contoh: *budget makan minggu ini 300rb*.")
            for m in st.session_state.messages:
                tampilkan_pesan(m)
            if prompt:
                pesan_user = {"role": "user", "content": prompt, "tools": []}
                tampilkan_pesan(pesan_user)
                with st.chat_message("assistant", avatar="💸"):
                    with st.spinner("Dompy lagi nyatet..."):
                        jawaban, calls, st.session_state.sesi, model = agent.tanya_dompy(
                            client, st.session_state.sesi, prompt, gaya, mode)
                pesan_dompy = {"role": "assistant", "content": jawaban, "tools": calls,
                               "model_cadangan": model if model and model != config.MODEL_NAME else None}
                st.session_state.messages += [pesan_user, pesan_dompy]
                st.rerun()  # gambar ulang supaya chat dan dashboard memakai data terbaru


def kartu_budget(s):
    persen = s["persen_terpakai"]
    warna = WARNA_LEVEL[s["level"]]
    if s["level"] == "lewat budget":
        info = f'<span style="color:{warna};font-weight:700">Lewat budget</span>'
    else:
        info = f'Sisa <b>{s["sisa"]}</b> · {s["jatah_per_hari"]}/hari'
    return (f'<div class="budget"><div class="atas"><div><span class="nama">{IKON.get(s["kategori"], "📦")} '
            f'{escape(s["kategori"])}</span><span class="periode">{escape(s["periode"])}</span></div>'
            f'<div class="nominal"><b>{s["terpakai"]}</b> / {s["budget"]}</div></div>'
            f'<div class="bar"><div style="width:{min(persen, 100):.0f}%;background:{warna}"></div></div>'
            f'<div class="bawah"><span>{info}</span><span>{persen:.0f}% · {s["hari_tersisa"]} hari lagi</span>'
            f'</div></div>')


with kanan:
    with st.container(key="kartu-budget"):
        st.html('<p class="judul-kartu">Budget aktif</p><p class="sub-kartu">Hijau aman, kuning waspada (≥75%), '
                'merah lewat budget.</p>')
        semua = sorted((status_budget(b) for b in budget_aktif()), key=lambda s: -s["persen_terpakai"])
        st.html("".join(kartu_budget(s) for s in semua) if semua else
                '<div class="kosong">Belum ada budget aktif.<br>Coba ketik: <i>budget makan minggu ini 300rb</i></div>')

    with st.container(key="kartu-grafik"):
        tab1, tab2, tab3 = st.tabs(["Per kategori", "30 hari terakhir", "Transaksi"])
        with tab1:
            st.plotly_chart(charts.grafik_kategori(), width="stretch", config={"displayModeBar": False})
        with tab2:
            st.plotly_chart(charts.grafik_harian(), width="stretch", config={"displayModeBar": False})
        with tab3:
            rows = daftar_transaksi(30)["transaksi"]
            if rows:
                df = pd.DataFrame(rows)[["id", "tanggal", "kategori", "deskripsi", "jumlah"]]
                df["tanggal"] = pd.to_datetime(df["tanggal"])
                st.dataframe(df, hide_index=True, width="stretch", height=280, column_config={
                    "id": st.column_config.NumberColumn("ID", width="small"),
                    "tanggal": st.column_config.DateColumn("Tanggal", format="DD MMM"),
                    "kategori": "Kategori", "deskripsi": "Keterangan", "jumlah": "Jumlah"})
            else:
                st.html('<div class="kosong">Belum ada transaksi.</div>')

## 8. Coba tools tanpa UI

Sel ini mengimpor modul yang baru ditulis dan memanggil beberapa tool langsung (tanpa Gemini), untuk memastikan database dan perhitungannya bekerja. `importlib.reload` dipakai supaya perubahan file langsung terbaca kalau sel di atas dijalankan ulang.

In [ ]:
import importlib
import json

import config, database, tools, agent, charts
for modul in (config, database, tools, agent, charts):
    importlib.reload(modul)

print(json.dumps(tools.cek_budget(), indent=2, ensure_ascii=False))
print(json.dumps(tools.ringkasan_pengeluaran(), indent=2, ensure_ascii=False))

Coba juga ngobrol dengan Dompy dari sini. Baris 🛠️ menunjukkan tool apa saja yang dipanggil Gemini untuk menjawab.

In [ ]:
client = agent.buat_client()
sesi = agent.buat_sesi_chat(client)
jawaban, calls, sesi, model = agent.tanya_dompy(client, sesi, "Halo Dompy! Kamu bisa bantu apa aja?")
for c in calls:
    print("🛠️", c)
print(f"[{model}]", jawaban)

### (Opsional) Isi data contoh

Untuk mencoba fitur evaluasi tanpa harus mencatat berminggu-minggu, ubah `ISI_DATA_CONTOH = True` lalu jalankan sel ini. Isinya pengeluaran acak yang realistis selama ±2 bulan terakhir plus beberapa budget. Biarkan `False` kalau mau mulai dari nol.

In [ ]:
import random
from datetime import timedelta

ISI_DATA_CONTOH = False


def isi_data_contoh(hari=60, seed=42):
    rng = random.Random(seed)
    katalog = {
        "Makanan": [("nasi padang", 22000), ("bakso", 18000), ("kopi susu", 25000), ("ayam geprek", 20000),
                    ("mie ayam", 15000), ("martabak", 35000), ("es teh", 5000), ("sarapan bubur", 12000)],
        "Transportasi": [("ojol", 18000), ("isi bensin", 30000), ("parkir", 5000), ("KRL", 6000)],
        "Belanja": [("sabun & sampo", 45000), ("belanja bulanan", 150000), ("kaos", 89000)],
        "Hiburan": [("nonton bioskop", 50000), ("top up game", 30000), ("langganan musik", 55000)],
    }
    today = database.hari_ini()
    with database.db() as conn:
        for i in range(hari, -1, -1):
            d = today - timedelta(days=i)
            weekend = d.weekday() >= 5
            for _ in range(rng.randint(2, 4) + weekend):
                kat = rng.choices(list(katalog), weights=[6, 3, 1, 1 + weekend * 2])[0]
                desk, harga = rng.choice(katalog[kat])
                harga = int(round(harga * rng.uniform(0.85, 1.25), -3))
                conn.execute("INSERT INTO expenses (category, amount, description, date, created_at) VALUES (?, ?, ?, ?, ?)",
                             (kat, harga, desk, d.isoformat(), database.sekarang()))
    tools.atur_budget("Makanan", 1500000, "bulanan")
    tools.atur_budget("Transportasi", 600000, "bulanan")
    tools.atur_budget("Hiburan", 250000, "bulanan")
    tools.atur_budget("Makanan", 350000, "mingguan")
    print("Data contoh berhasil diisi.")


if ISI_DATA_CONTOH:
    isi_data_contoh()

## 9. Jalankan Dompy

- **Di Colab**: sel ini menjalankan Streamlit di belakang layar lalu membuat link publik `trycloudflare.com` (lewat Cloudflare Tunnel) yang bisa dibuka di browser maupun HP. Link berlaku selama sel/runtime masih hidup.
- **Di PC (Jupyter/VS Code)**: Streamlit dijalankan dan browser terbuka ke `http://localhost:8501`. Bisa juga tanpa notebook: `streamlit run app.py` dari terminal.

Untuk menghentikan aplikasi, jalankan sel paling bawah.

In [ ]:
import re
import subprocess
import sys
import time
import urllib.request
import webbrowser

PORT = 8501

app_log = open("streamlit.log", "w")
app_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", str(PORT)],
                            stdout=app_log, stderr=subprocess.STDOUT)
time.sleep(5)

if IN_COLAB:
    if not os.path.exists("cloudflared"):
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
        os.chmod("cloudflared", 0o755)
    tunnel_log = open("tunnel.log", "w")
    tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                                   stdout=tunnel_log, stderr=subprocess.STDOUT)
    url = None
    for _ in range(30):
        time.sleep(1)
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
        if found:
            url = found.group(0)
            break
    print("🌐 Buka Dompy di:", url or "(link belum muncul, cek file tunnel.log)")
else:
    url = f"http://localhost:{PORT}"
    webbrowser.open(url)
    print("🌐 Dompy berjalan di:", url)

In [ ]:
# Hentikan aplikasi
app_proc.terminate()
if IN_COLAB:
    tunnel_proc.terminate()
print("Dompy dihentikan.")